In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split

In [2]:
nltk.download('stopwords') #download all the 
nltk.download('punkt')
STOPWORDS = set(stopwords.words('english'))
print("✅ All imports successful!")
print(f"Number of stopwords: {len(STOPWORDS)}")

✅ All imports successful!
Number of stopwords: 198


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\KIIT0001\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\KIIT0001\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
#importing the datasets
true_df  = pd.read_csv('True.csv')

In [4]:
fake_df  = pd.read_csv('Fake.csv')

In [5]:
true_df['label'] = 0   # 0 = Real
fake_df['label'] = 1   # 1 = Fake

In [6]:
print(true_df.columns.tolist())

['title', 'text', 'subject', 'date', 'label']


In [7]:
# Combine title + text into one field so that later on we can use it as content
true_df['content'] = true_df['title'] + " " + true_df['text']
fake_df['content'] = fake_df['title'] + " " + fake_df['text']

In [8]:
true_df.head()

,title,text,subject,date,label,content
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",0,"As U.S. budget fight looms, Republicans flip t..."
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",0,U.S. military to accept transgender recruits o...
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",0,Senior U.S. Republican senator: 'Let Mr. Muell...
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",0,FBI Russia probe helped by Australian diplomat...
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",0,Trump wants Postal Service to charge 'much mor...


In [9]:
print(true_df['content'].head)

<bound method NDFrame.head of 0        As U.S. budget fight looms, Republicans flip t...
1        U.S. military to accept transgender recruits o...
2        Senior U.S. Republican senator: 'Let Mr. Muell...
3        FBI Russia probe helped by Australian diplomat...
4        Trump wants Postal Service to charge 'much mor...
                               ...                        
21412    'Fully committed' NATO backs new U.S. approach...
21413    LexisNexis withdrew two products from Chinese ...
21414    Minsk cultural hub becomes haven from authorit...
21415    Vatican upbeat on possibility of Pope Francis ...
21416    Indonesia to buy $1.14 billion worth of Russia...
Name: content, Length: 21417, dtype: object>


In [10]:
#this is to concat the true and fake samples
df = pd.concat([true_df, fake_df], ignore_index=True)
#this is to randomize the concatation
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Total samples : {len(df)}")
print(f"Fake articles : {df['label'].sum()}")
print(f"Real articles : {len(df) - df['label'].sum()}")

Total samples : 44898
Fake articles : 23481
Real articles : 21417


In [11]:
#cleaning the text
def clean_text(text):
    
    # Lowercase everything. "news" and "News" are same so we change all of the text in lowercase
    text = text.lower()
    
    # Remove URLs (http://... or www....) urls are not needed to understand the fake and real news
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # Remove HTML tags like <br>, <p>...like who needs that?
    text = re.sub(r'<.*?>', '', text)
    
    # Remove punctuation & special characters
    # Keep only letters and spaces
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Remove extra space. it only takes memory and my precious time
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Remove stopwords 
    tokens = text.split() #splits the senetence into tokens. like "Hello this is Google" will now be ["hello", "this", "is", "Google"]
    
    new_tokens = []

    #used to remove stop words. 
    #the is a an in on and of to etc are stop words that are already in built in the STOPWORD collection which we got when we wrote from nltk.corpus import stopwords

    for word in tokens:
        if word not in STOPWORDS and len(word) > 2:
            new_tokens.append(word)

    tokens = new_tokens
    
    return ' '.join(tokens)

In [12]:
#apply the fucntion to thte entire dataset 

df['clean_content'] = df['content'].apply(clean_text)

# check if this worked
idx = 3 #for content in first/second/third whatever row
print("BEFORE:", df['content'][idx][:30]) #checking content from idx row and only the first 20 characters (0-19th) 
print()

print("AFTER :", df['clean_content'][idx][:30])

#see how AG was removed, to was removed and everything was in lower case 

BEFORE: California AG pledges to defen

AFTER : california pledges defend birt


In [13]:
texts  = df['clean_content'].tolist()

In [14]:
labels = df['label'].tolist()

In [15]:
# First split: 70% train, 30% temp

X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels,
    test_size=0.30,
    random_state=42,
    stratify=labels       # ← maintains class ratio in each split
)

In [16]:
# Second split: split temp 50/50 → 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [17]:
print(f"Training samples   : {len(X_train)}")   # ~31,428
print(f"Validation samples : {len(X_val)}")     # ~6,735
print(f"Test samples       : {len(X_test)}")    # ~6,735

Training samples   : 31428
Validation samples : 6735
Test samples       : 6735


In [18]:
# Verify class balance in each split
import collections
print("Train label dist:", collections.Counter(y_train))
print("Val   label dist:", collections.Counter(y_val))

Train label dist: Counter({1: 16436, 0: 14992})
Val   label dist: Counter({1: 3522, 0: 3213})


In [19]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.10.0+cpu
False


In [20]:
from transformers import BertTokenizer

In [22]:
pip install torch transformers datasets scikit-learn pandas

   ---------------------------------------- 0.0/527.5 kB ? eta -:--:--
   ---------------------------------------- 527.5/527.5 kB 5.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/27.5 MB ? eta -:--:--
   ------ --------------------------------- 4.2/27.5 MB 21.8 MB/s eta 0:00:02
   --------- ------------------------------ 6.3/27.5 MB 15.0 MB/s eta 0:00:02
   ------------ --------------------------- 8.4/27.5 MB 13.2 MB/s eta 0:00:02
   -------------- ------------------------- 10.2/27.5 MB 12.3 MB/s eta 0:00:02
   ----------------- ---------------------- 11.8/27.5 MB 11.1 MB/s eta 0:00:02
   ------------------- -------------------- 13.6/27.5 MB 10.6 MB/s eta 0:00:02
   ---------------------- ----------------- 15.5/27.5 MB 10.2 MB/s eta 0:00:02
   ------------------------- -------------- 17.3/27.5 MB 10.2 MB/s eta 0:00:02
   ---------------------------- ----------- 19.4/27.5 MB 10.1 MB/s eta 0:00:01
   ------------------------------- -------- 21.8/27.5 MB 10.1 MB/s eta 

In [23]:
# Option A: BERT Tokenizer (for BERT model — recommended)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
# Tokenize a sample
sample = "president claims election fraud rigged votes"
tokens = tokenizer.tokenize(sample)
ids    = tokenizer.convert_tokens_to_ids(tokens)
print("Tokens:", tokens)
# ['president', 'claims', 'election', 'fraud', 'rig', '##ged', 'votes']
print("IDs   :", ids)
# [2343, 4447, 2602, 5480, 15461, 4775, 5191]
# Full encoding with padding & truncation
encoding = tokenizer(
    sample,
    max_length=512,
    padding='max_length',     # pad short texts with [PAD] tokens
    truncation=True,          # cut long texts at 512 tokens
    return_tensors='pt'       # return PyTorch tensors
)
print("input_ids shape     :", encoding['input_ids'].shape)
print("attention_mask shape:", encoding['attention_mask'].shape)
# Both: torch.Size([1, 512])

Tokens: ['president', 'claims', 'election', 'fraud', 'rigged', 'votes']
IDs   : [2343, 4447, 2602, 9861, 25216, 4494]
input_ids shape     : torch.Size([1, 512])
attention_mask shape: torch.Size([1, 512])


In [24]:
import torch
from torch.utils.data import Dataset, DataLoader    #dataset is for one sample and dataloader is for batch of data to be trained. 
class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts     = texts       # list of cleaned article strings
        self.labels    = labels      # list of 0/1 integers
        self.tokenizer = tokenizer
        self.max_len   = max_len
    def __len__(self):
        # Called as: len(dataset) → returns 31428 for training set
        return len(self.texts)
    def __getitem__(self, idx):
        # Called as: dataset[5] → returns sample #5
        text  = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'     : encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label'         : torch.tensor(label, dtype=torch.long)
        }
# Create datasets
train_dataset = FakeNewsDataset(X_train, y_train, tokenizer)
val_dataset   = FakeNewsDataset(X_val,   y_val,   tokenizer)
test_dataset  = FakeNewsDataset(X_test,  y_test,  tokenizer)
# Create DataLoaders — these handle batching automatically
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)
print(f"Batches per epoch: {len(train_loader)}")  # 31428 / 16 ≈ 1964

Batches per epoch: 1965
